# Obtain and verify the eThekwini municipal boundary

This notebook retrieves a public municipal snapshot, checks identity, geometry and CRS, and writes a verification report and local QGIS layer. It does not verify the boundary’s legal effective date or calculate flood exposure. Existing raw snapshots are reused, not overwritten.

Run in the project research environment; a new checkout requires internet access for the download cell.

In [ ]:
from pathlib import Path
import json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout.")
RAW = ROOT / "data/raw/ethekwini_boundary"
RAW.mkdir(parents=True, exist_ok=True)


In [ ]:
from urllib.request import urlopen
from urllib.parse import urlencode
from datetime import datetime, timezone

BASE = "https://services3.arcgis.com/HO0zfySJshlD6Twu/arcgis/rest/services/eThekwini_Municipal_Boundary/FeatureServer/0"
ITEM = "https://www.arcgis.com/sharing/rest/content/items/21e2dd4e945840779308aaf1dd9f6ac6?f=pjson"
requests = {
    "layer.json": BASE + "?f=pjson",
    "item.json": ITEM,
    "metadata.xml": BASE + "/metadata?f=xml",
    "count.json": BASE + "/query?" + urlencode({"where": "1=1", "returnCountOnly": "true", "f": "json"}),
    "boundary_native.json": BASE + "/query?" + urlencode({"where": "1=1", "outFields": "*", "returnGeometry": "true", "f": "json"}),
    "boundary_wgs84.geojson": BASE + "/query?" + urlencode({"where": "1=1", "outFields": "*", "returnGeometry": "true", "outSR": "4326", "f": "geojson"}),
}
# Reuse the retained snapshot. To examine a newer release, use a new directory
# and dataset ID; never silently replace a registered source snapshot.
manifest_path = RAW / "download_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
for filename, url in requests.items():
    destination = RAW / filename
    if destination.exists():
        continue
    with urlopen(url, timeout=45) as response:
        content = response.read()
    if filename.endswith((".json", ".geojson")):
        payload = json.loads(content)
        if "error" in payload:
            raise RuntimeError(payload["error"])
    destination.write_bytes(content)
    manifest.append({"file": filename, "url": url,
                     "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Source files available:", len(requests))


In [ ]:
import hashlib
import platform
import geopandas as gpd
import shapely
import pyproj
from shapely.geometry import Point, Polygon, shape
from pyproj import CRS, Transformer, Geod

layer = json.loads((RAW / "layer.json").read_text())
item = json.loads((RAW / "item.json").read_text())
native = json.loads((RAW / "boundary_native.json").read_text())
collection = json.loads((RAW / "boundary_wgs84.geojson").read_text())
assert json.loads((RAW / "count.json").read_text())["count"] == 1
assert not native.get("exceededTransferLimit", False)
assert len(native["features"]) == len(collection["features"]) == 1
assert item["access"] == "public" and item["owner"] == "opendata_ethekwini"
attributes = native["features"][0]["attributes"]
assert attributes["CODE"] == "ETH" and attributes["MUNICNAME"] == "eThekwini"
geometry = shape(collection["features"][0]["geometry"])
assert geometry.geom_type == "Polygon" and geometry.is_valid and not geometry.is_empty

# The metadata's EPSG code 0 is not a usable CRS identifier. Parse the actual
# WKT supplied by the feature query; do not substitute a similarly named CRS.
native_crs = CRS.from_wkt(native["spatialReference"]["wkt"])
assert native_crs.is_projected
rings = native["features"][0]["geometry"]["rings"]
assert len(rings) == 1, "Review ring topology if the source structure changes."
transformer = Transformer.from_crs(native_crs, "EPSG:4326", always_xy=True)
transformed = Polygon([transformer.transform(x, y) for x, y in rings[0]])
transform_difference = geometry.hausdorff_distance(transformed)
assert transform_difference < 1e-8, "Server and local CRS transformations disagree."

frame = gpd.read_file(RAW / "boundary_wgs84.geojson")
assert frame.crs.to_epsg() == 4326
bounds = list(geometry.bounds)
assert 30 < bounds[0] < bounds[2] < 32 and -31 < bounds[1] < bounds[3] < -29
location_checks = {
    "Durban centre inside": geometry.covers(Point(31.0218, -29.8587)),
    "Cape Town outside": not geometry.covers(Point(18.4241, -33.9249)),
    "Pietermaritzburg outside": not geometry.covers(Point(30.3794, -29.6006)),
}
assert all(location_checks.values())
geodesic_area_km2 = abs(Geod(ellps="WGS84").geometry_area_perimeter(geometry)[0]) / 1e6
assert 2400 < geodesic_area_km2 < 2700, "Unexpected extent: investigate before use."

# UTM 36S is suitable for metre-based screening at this location. Its area is
# not identical to ellipsoidal/geodesic area because of projection distortion.
projected = frame.to_crs("EPSG:32736")
output_dir = ROOT / "data/processed/ethekwini"
output_dir.mkdir(parents=True, exist_ok=True)
projected.to_file(output_dir / "municipal_boundary.gpkg", layer="municipal_boundary", driver="GPKG")
roundtrip = gpd.read_file(output_dir / "municipal_boundary.gpkg")
assert roundtrip.crs.to_epsg() == 32736 and roundtrip.geometry.is_valid.all()
assert roundtrip.geometry.iloc[0].equals_exact(projected.geometry.iloc[0], 1e-6)

report = {
    "dataset_id": "ethekwini_boundary_arcgis_2022_snapshot_20261020",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_layer_url": BASE,
    "source_item_id": item["id"],
    "publisher": item["owner"],
    "source_credit": item["accessInformation"],
    "feature_count": len(frame),
    "municipality_code": attributes["CODE"],
    "geometry_type": geometry.geom_type,
    "geometry_valid": bool(geometry.is_valid),
    "bounds_wgs84": bounds,
    "native_crs_wkt": native_crs.to_wkt(),
    "download_crs": "EPSG:4326",
    "analysis_crs": "EPSG:32736",
    "native_to_server_hausdorff_degrees": transform_difference,
    "geodesic_area_km2": geodesic_area_km2,
    "utm_area_km2": float(projected.area.sum() / 1e6),
    "location_checks": {k: bool(v) for k, v in location_checks.items()},
    "service_data_last_edit_utc": datetime.fromtimestamp(layer["editingInfo"]["dataLastEditDate"] / 1000, timezone.utc).isoformat(),
    "item_modified_utc": datetime.fromtimestamp(item["modified"] / 1000, timezone.utc).isoformat(),
    "effective_boundary_date": None,
    "metadata_use_limitations": item.get("licenseInfo"),
    "verification_scope": "Publisher identity, municipality identity, completeness, CRS transformation, geometry, and spatial sanity checks. Legal effective date and currentness remain unverified.",
    "redistribution_permission": "No explicit open redistribution license identified in inspected metadata; source geometry remains local and untracked.",
    "checksums_sha256": {name: hashlib.sha256((RAW / name).read_bytes()).hexdigest() for name in requests},
    "environment": {"python": platform.python_version(), "geopandas": gpd.__version__, "shapely": shapely.__version__, "pyproj": pyproj.__version__},
}
report_path = ROOT / "metadata/ethekwini_boundary_verification.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps({k: report[k] for k in ["geometry_valid", "bounds_wgs84", "geodesic_area_km2", "utm_area_km2", "location_checks", "effective_boundary_date"]}, indent=2))
print("Verification report:", report_path.relative_to(ROOT))


## Interpretation

Read `research/boundary_verification.md` and the generated JSON report. February 2022 service metadata does not prove the legal boundary in April 2022 or today. The geometry is suitable for provisional pilot screening, with this limitation retained. Local artifacts are Git-ignored and must be retained in persistent storage. Re-fetching a live service later may produce different bytes; compare hashes with the registered snapshot before treating it as the same input.